# 01 — Data Cleaning & Preprocessing

A compact checklist for a real tabular workflow: inspect the data, fix types, remove duplicates, separate features/target, split **before** learning preprocessing values, then fit imputation/scaling/encoding only on the training data.

In [ ]:
import numpy as np
import pandas as pd

raw = pd.DataFrame({
    'age': [25, 31, 31, 44, np.nan, 52, 29, 38, 41, 35],
    'income': ['32000', '41000', '41000', '62000', '50000', 'not_known', '37000', '48000', '250000', '45000'],
    'city': ['London', 'London', 'London', 'Leeds', 'Leeds', None, 'London', 'Bristol', 'Bristol', 'Leeds'],
    'signup_date': ['2026-01-05', '2026-01-08', '2026-01-08', '2026-02-01', 'bad_date', '2026-02-10', '2026-03-02', '2026-03-11', '2026-03-20', '2026-04-01'],
    'converted': [0, 1, 1, 1, 0, 1, 0, 1, 1, 0],
})
raw

In [ ]:
# 1. Audit before changing anything
audit = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing': raw.isna().sum(),
    'unique': raw.nunique(dropna=False),
})
print('Exact duplicate rows:', raw.duplicated().sum())
display(audit)

# 2. Remove exact duplicates and fix parseable types
df = raw.drop_duplicates().copy()
df['income'] = pd.to_numeric(df['income'], errors='coerce')
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
df['signup_month'] = df['signup_date'].dt.month
df = df.drop(columns='signup_date')

print(df.isna().sum())
df.head()

## Leakage-safe preprocessing

Do not calculate medians, means, standard deviations or category vocabularies on the complete dataset before the split. Put those operations inside a scikit-learn pipeline and fit it on training rows only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df.drop(columns='converted')
y = df['converted']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric = ['age', 'income', 'signup_month']
categorical = ['city']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical),
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print('Train shape:', X_train_ready.shape)
print('Test shape:', X_test_ready.shape)

## What I would check next

- target balance and whether stratification/grouping/time order is appropriate
- impossible values and domain-specific ranges rather than deleting every statistical outlier
- leakage/proxy columns that reveal the target
- train/test drift and unseen categories
- a baseline model before adding complexity
- reproducible data-quality counts in the final report